In [ ]:
import torch
import torchvision
import numpy as np
import torchvision.transforms as transforms
from torch.utils.data import Subset, DataLoader, Dataset
import sys
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from tqdm.notebook import tqdm
from google.colab import files
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


# Função para dividir entre rotulado e não rotulado

---

Função que vamos usar, onde recebemos o dataset inteiro de treino e o número de imagens supervisionadas por classe e dividimos em subsets

In [ ]:
def create_ssl_split(full_dataset, num_labeled_per_class):
    """
    Divide o dataset de treino (50k) em rotulado e não-rotulado.
    Retorna dois OBJETOS SUBSET.
    """
    try:
        targets = np.array(full_dataset.targets)
        num_classes = len(np.unique(targets))
    except AttributeError:
        raise ValueError("Dataset must have a .targets attribute.")

    labeled_indices = []
    unlabeled_indices = []

    for c in range(num_classes):
        class_indices = np.where(targets == c)[0]

        # Verifica se há amostras suficientes no conjunto de 50k
        if len(class_indices) < num_labeled_per_class:
            raise ValueError(f"Class {c} has {len(class_indices)} samples, "
                             f"but {num_labeled_per_class} were requested.")

        np.random.shuffle(class_indices)
        labeled_indices.extend(class_indices[:num_labeled_per_class])
        unlabeled_indices.extend(class_indices[num_labeled_per_class:])

    # Usando Subset, que vem do torch. Só usa on indices
    labeled_subset = Subset(full_dataset, labeled_indices)

    # Usamos todas as imagens, mas sem o label agora
    unlabeled_indices = list(unlabeled_indices) + list(labeled_indices)
    unlabeled_subset = Subset(full_dataset, unlabeled_indices)

    return labeled_subset, unlabeled_subset

# Dataset com Imagens Supervisionadas e Não Supervisionadas

---

Pega o itme dependendo do 'modo' que você passar, podendo retornar uma imagem e um label ou duas imagens, com augmentações diferentes

In [ ]:
class FixMatchDataset(Dataset):
    """
    Um Dataset para o FixMatch dividido em dois

    - 'labeled': Retorna (weak_image, label)
    - 'unlabeled': Retorna (weak_image, strong_image)
    """
    def __init__(self, subset, mode, transform_weak, transform_strong=None):
        self.subset = subset
        self.mode = mode
        self.transform_weak = transform_weak
        self.transform_strong = transform_strong

        if self.mode not in ['labeled', 'unlabeled']:
            raise ValueError("'Mode' precisa ser 'labeled' ou 'unlabeled'.")
        if self.mode == 'unlabeled' and self.transform_strong is None:
            raise ValueError("Caso 'mode' seja 'unlabeled', precisa de uma transformação forte")

    def __getitem__(self, index):
        # Pega a imagem e o label
        image, label = self.subset[index]

        if self.mode == 'labeled':
            img_weak = self.transform_weak(image)
            return img_weak, label

        else: # mode == 'unlabeled'
            img_weak = self.transform_weak(image)
            img_strong = self.transform_strong(image)
            return img_weak, img_strong

    def __len__(self):
        return len(self.subset)

# Dataloaders e Transformações

---

Aqui definimos as transformações, baixamos os datasets do CIFAR10 direto do torch e definimos funções de 'get' para usar as imagens de treino, validação e teste

In [ ]:
class CIFAR10DataModule:
    """
    Gerencia os dados.
    - Treino (50k) -> Rotulado (X) e Não-Rotulado (50k - X)
    - Teste (10k)  -> Validação (5k) e Teste (5k)
    """
    def __init__(self, data_dir='./data',
                 num_labeled_per_class=25,
                 test_split_size=5000, # Define o tamanho do teste final
                 batch_size=64,
                 unlabeled_ratio=7,    # Ratio definido no paper para imagens unlabeled
                 num_workers=2):

        self.data_dir = data_dir
        self.num_labeled_per_class = num_labeled_per_class
        self.test_split_size = test_split_size
        self.batch_size = batch_size
        self.unlabeled_ratio = unlabeled_ratio
        self.num_workers = num_workers

        cifar_mean = (0.4914, 0.4822, 0.4465)
        cifar_std = (0.2023, 0.1994, 0.2010)

        # Transformação simples, usada na labeled e unlabeled
        self.transform_weak = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.ToTensor(),
            transforms.Normalize(cifar_mean, cifar_std)
        ])

        # Transformação forte, usada só nas unlabeled
        self.transform_strong = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandAugment(),
            transforms.ToTensor(),
            transforms.Normalize(cifar_mean, cifar_std)
        ])

        # Transformação para evaluation, passando pra tensor e normalizando
        self.transform_eval = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(cifar_mean, cifar_std)
        ])

    def setup(self):
        """
        Baixa os dados e cria as divisões.
        """
        # Carrega o dataset de TREINO (50k)
        full_train_dataset = torchvision.datasets.CIFAR10(
            root=self.data_dir,
            train=True,
            download=True,
            transform=None # Transfs aplicadas depois
        )

        # Carrega o dataset de TESTE (10k)
        full_test_dataset = torchvision.datasets.CIFAR10(
            root=self.data_dir,
            train=False,
            download=True,
            transform=None
        )

        # DIVISÃO VALIDAÇÃO / TESTE (do conjunto de 10k)
        test_targets = np.array(full_test_dataset.targets)
        test_indices = np.arange(len(test_targets))

        # test_size=5000, e val_size=5000 (defininimos 5000)
        val_indices, test_indices = train_test_split(
            test_indices,
            test_size=self.test_split_size,
            stratify=test_targets,
            random_state=42
        )

        # Cria o dataset de validação (5k)
        val_subset = Subset(full_test_dataset, val_indices)
        self.val_dataset = FixMatchDataset(
            subset=val_subset,
            mode='labeled',
            transform_weak=self.transform_eval # Usa a transform de avaliação
        )

        # Cria o dataset de teste (5k)
        test_subset = Subset(full_test_dataset, test_indices)
        self.test_dataset = FixMatchDataset(
            subset=test_subset,
            mode='labeled',
            transform_weak=self.transform_eval # Usa a transform de avaliação
        )

        # DIVISÃO ROTULADO / NÃO-ROTULADO (dos 50k de treino)
        # Chama a função de split
        labeled_subset, unlabeled_subset = create_ssl_split(
            full_train_dataset,
            self.num_labeled_per_class
        )

        # Cria os datasets com labels
        self.train_labeled_dataset = FixMatchDataset(
            subset=labeled_subset,
            mode='labeled',
            transform_weak=self.transform_weak
        )

        # sem labels
        self.train_unlabeled_dataset = FixMatchDataset(
            subset=unlabeled_subset,
            mode='unlabeled',
            transform_weak=self.transform_weak,
            transform_strong=self.transform_strong
        )

    # Retornam os loaders com labels e sem
    def get_train_loaders(self):
        labeled_loader = DataLoader(
            self.train_labeled_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            drop_last=False
        )
        unlabeled_loader = DataLoader(
            self.train_unlabeled_dataset,
            batch_size=self.batch_size * self.unlabeled_ratio,
            shuffle=True,
            num_workers=self.num_workers,
            drop_last=True
        )
        return labeled_loader, unlabeled_loader

    def get_val_loader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size * 2,
            shuffle=False,
            num_workers=self.num_workers
        )

    def get_test_loader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size * 2,
            shuffle=False,
            num_workers=self.num_workers
        )

# Modelo e Avaliação

---

### Resnet-18

Usamos a resnet-18, que vem pronta do torch, apenas modificando as 2 primeiras camadas, para evitar uma diminuição de resolução, já que a imagem do CIFAR10 é muito pequena, 32x32.

In [ ]:
def create_model():
    """
    Cria e retorna uma nova instância da ResNet-18 modificada para CIFAR-10.
    """
    model = models.resnet18(weights=None, num_classes=10)

    # Como a imagem é pequena, fazemos a primeira convolução e maxpool não diminuir a resolução
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def evaluate_model(model, val_loader, criterion, device):
    """
    Avalia a loss média e a acurácia do modelo no conjunto de validação/teste.
    """
    model.eval() # modelo em modo de avaliação
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            # Calcula a loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Calcula a acurácia
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(val_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# TREINO DO MODELO

Faz o treino supervisionado e não supervisionado ao mesmo tempo, usando as informações dos dados com label para "chutar" bons pseudo-labels nos dados não supervisionados e ir tentando melhorar a acurácia

---

### Treino Supervisionado

Usa um Cross-Entropy normal, faz a classificação comum de modelos como esse

### Treino Não Supervisionado

Aplica uma aumentação fraca e forte em duas imagens, e caso a probabilidade de certeza do modelo for alta (maior que 0.95 ou 0.99) usa a diferença entre as respostas do modelo para essas 2 aumentações para punir esses erros de consistência.

In [ ]:
def train_model(model, labeled_loader, criterion_supervised, unlabeled_loader, criterion_unsupervised,
                optimizer, num_epochs, TAU, LAMBDA_U, val_loader,
                device=device):

    model.to(device)
    history = {
        's_loss': [], # supervised
        'uns_loss': [], # unsupervised
        'total_loss': [],
        'val_loss': [],
        'val_accuracy': []
    }

    print(f"Iniciando treinamento FixMatch com otimizador: {optimizer.__class__.__name__}")

    epoch_bar = tqdm(range(num_epochs), desc="Progresso Total (FixMatch)")

    for epoch in epoch_bar:
        model.train()
        labeled_iter = iter(labeled_loader)
        unlabeled_iter = iter(unlabeled_loader)
        num_steps_per_epoch = len(unlabeled_loader)

        # Ls -> Loss da supervised, Lu -> Loss unsupervised
        Ls_epoch_total = 0.0
        Lu_epoch_total = 0.0

        step_bar = tqdm(
            range(num_steps_per_epoch),
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            leave=False
        )

        for step in step_bar:
            # pega as imagems e labels da parte supervisionada
            try:
                images_labeled, labels = next(labeled_iter)
            except StopIteration:
                labeled_iter = iter(labeled_loader)
                images_labeled, labels = next(labeled_iter)

            # pega as imagems com augmentation fraca e forte da parte não supervisionada
            try:
                images_unlabeled_weak, images_unlabeled_strong = next(unlabeled_iter)
            except StopIteration:
                unlabeled_iter = iter(unlabeled_loader)
                images_unlabeled_weak, images_unlabeled_strong = next(unlabeled_iter)

            # passando tudo pro CUDA
            images_labeled = images_labeled.to(device)
            labels = labels.to(device)
            images_unlabeled_weak = images_unlabeled_weak.to(device)
            images_unlabeled_strong = images_unlabeled_strong.to(device)

            outputs_labeled = model(images_labeled)
            Ls = criterion_supervised(outputs_labeled, labels)

            # não modifica os parâmetros por agora, só calcula as probabilidades
            with torch.no_grad():
                outputs_unlabeled_weak = model(images_unlabeled_weak)
                probs_weak = torch.softmax(outputs_unlabeled_weak, dim=1)
                max_probs, pseudo_labels = torch.max(probs_weak, dim=1)

                # se a mask for maior que o TAU, mantem o pseudo-label
                mask = max_probs.ge(TAU).float()

            outputs_unlabeled_strong = model(images_unlabeled_strong)
            Lu_per_sample = criterion_unsupervised(outputs_unlabeled_strong, pseudo_labels)
            Lu_masked = Lu_per_sample * mask
            Lu = Lu_masked.sum() / (mask.sum() + 1e-10) # Critério Unsupervised
            Lu_final = (LAMBDA_U * Lu)

            Ls_epoch_total += Ls.item()
            Lu_epoch_total += Lu_final.item()

            L_total = Ls + Lu_final
            optimizer.zero_grad()
            L_total.backward()
            optimizer.step()

            step_bar.set_postfix(
                Ls_batch=f"{Ls.item():.4f}",
                Lu_batch=f"{Lu_final.item():.4f}"
            )

        # VALIDAÇÃO (NO FIM DA ÉPOCA)
        val_loss, val_acc = evaluate_model(model, val_loader, criterion_supervised, device)

        # Salva as médias da época
        history['s_loss'].append(Ls_epoch_total / num_steps_per_epoch)
        history['uns_loss'].append(Lu_epoch_total / num_steps_per_epoch)
        history['total_loss'].append((Ls_epoch_total + Lu_epoch_total) / num_steps_per_epoch)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        # Atualiza a barra de épocas com a acurácia de validação
        epoch_bar.set_postfix(
            Avg_Ls=f"{(Ls_epoch_total / num_steps_per_epoch):.4f}",
            Avg_Lu=f"{(Lu_epoch_total / num_steps_per_epoch):.4f}",
            Val_Acc=f"{val_acc:.2f}%" # <-- Acurácia de validação
        )

    print(f"\nTreinamento FixMatch concluído.")
    return history

# Treino Supervisionado (Baseline)

Faz o treino só com as imagens com label (no caso, se só tivermos 25 por classe, só treina com 250). Permite vermos se o FixMatch está melhorando ou não a acurácia do modelo.

In [ ]:
def train_supervised_only(model, unlabeled_loader, labeled_loader, criterion_supervised,
                          optimizer, num_epochs,
                          val_loader,
                          device=device):

    model.to(device)
    history = {
        's_loss': [],
        'val_loss': [],
        'val_accuracy': []
    }

    print(f"Iniciando treinamento Baseline (Apenas Supervisionado)...")

    epoch_bar = tqdm(range(num_epochs), desc="Progresso Total (Baseline)")

    num_steps_per_epoch = len(unlabeled_loader)

    if len(labeled_loader) == 0:
        print("Aviso: Labeled loader com 0 passos. Treino baseline não executado.")
        return history

    for epoch in epoch_bar:
        model.train()
        Ls_epoch_total = 0.0

        labeled_iter = iter(labeled_loader)

        step_bar = tqdm(
            range(num_steps_per_epoch),
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            leave=False
        )

        for step in step_bar:
            try:
                images_labeled, labels = next(labeled_iter)
            except StopIteration:
                labeled_iter = iter(labeled_loader)
                images_labeled, labels = next(labeled_iter)

            images_labeled = images_labeled.to(device)
            labels = labels.to(device)

            outputs_labeled = model(images_labeled)
            Ls = criterion_supervised(outputs_labeled, labels)

            Ls_epoch_total += Ls.item()

            optimizer.zero_grad()
            Ls.backward()
            optimizer.step()

            step_bar.set_postfix(Ls_batch=f"{Ls.item():.4f}")

        val_loss, val_acc = evaluate_model(model, val_loader, criterion_supervised, device)

        avg_epoch_loss = Ls_epoch_total / num_steps_per_epoch
        history['s_loss'].append(avg_epoch_loss)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        epoch_bar.set_postfix(
            Avg_Ls=f"{avg_epoch_loss:.4f}",
            Val_Acc=f"{val_acc:.2f}%"
        )

    print(f"\nTreinamento Baseline concluído.")
    return history

# Plot das Losses

In [ ]:
def plot_history(history, title):
    """
    Plota a loss de treino/validação e a acurácia de validação.
    """
    # 2 subplots (1 linha, 2 colunas)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(title, fontsize=16)

    # --- Gráfico 1: Loss ---
    ax1.plot(history['s_loss'], label='Loss Sup. (Treino)', color='blue', linestyle='--')
    ax1.plot(history['val_loss'], label='Loss (Validação)', color='orange')

    # Adiciona a loss não-supervisionada se for FixMatch
    if 'uns_loss' in history:
        ax1.plot(history['uns_loss'], label='Loss Não-Sup. (Treino)', color='green', linestyle=':')

    ax1.set_title('Loss ao longo das Épocas')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # --- Gráfico 2: Acurácia ---
    ax2.plot(history['val_accuracy'], label='Acurácia (Validação)', color='red')
    ax2.set_title('Acurácia ao longo das Épocas')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Acurácia (%)')
    ax2.legend()
    ax2.grid(True)

    plt.show()

# Hiperparâmetros

In [ ]:
# Hiperparâmetros do FixMatch
NUM_EPOCHS = 10                 # Número de épocas para cada experimento
LEARNING_RATE = 0.01           # LR comum
TAU = 0.95                     # Limiar de confiança
LAMBDA_U = 1.0                 # Peso da loss não-supervisionada
LABEL_COUNTS = [1, 4, 25, 400] # Experimentos a serem rodados (rótulos por classe)

# Loss functions
loss_fn_supervised = nn.CrossEntropyLoss(reduction='mean')
loss_fn_unsupervised = nn.CrossEntropyLoss(reduction='none') # 'none' para poder comparar nas unlabeled

# Setup dos Dados

---

Aqui usamos o nosso CIFAR10DataModule e salvamos todos esses loaders e datasets dentro de um dicionário, que vamos usar para fazer o treino do modelo logo abaixo dessa célula.

In [ ]:
data_setups = {}

print("Preparando todos os conjuntos de dados (Teste 10k -> Val 5k / Teste 5k)...")
print("=" * 40)

for num_labels in LABEL_COUNTS:
    print(f"\n--- Preparando setup para: {num_labels} RÓTULOS POR CLASSE ---")

    data_module = CIFAR10DataModule(
        num_labeled_per_class=num_labels,
        test_split_size=5000,
        batch_size=64,
        unlabeled_ratio=7,
        num_workers=2
    )

    data_module.setup()

    labeled_loader, unlabeled_loader = data_module.get_train_loaders()
    val_loader = data_module.get_val_loader()
    test_loader = data_module.get_test_loader()

    print(f"Setup completo para {num_labels} rótulos.")
    print(f"  Imagens de Treino (Rotuladas): {len(data_module.train_labeled_dataset)}")
    print(f"  Imagens de Treino (Não-Rotuladas): {len(data_module.train_unlabeled_dataset)}")
    print(f"  Imagens de VALIDAÇÃO: {len(data_module.val_dataset)}")   # Deve ser 5000
    print(f"  Imagens de TESTE: {len(data_module.test_dataset)}")      # Deve ser 5000

    # Armazena todos os loaders
    data_setups[num_labels] = {
        'labeled_loader': labeled_loader,
        'unlabeled_loader': unlabeled_loader,
        'val_loader': val_loader,
        'test_loader': test_loader
    }

print("\n" + "=" * 40)

Preparando todos os conjuntos de dados (Teste 10k -> Val 5k / Teste 5k)...

--- Preparando setup para: 1 RÓTULOS POR CLASSE ---


100%|██████████| 170M/170M [00:19<00:00, 8.68MB/s]


Setup completo para 1 rótulos.
  Imagens de Treino (Rotuladas): 10
  Imagens de Treino (Não-Rotuladas): 50000
  Imagens de VALIDAÇÃO: 5000
  Imagens de TESTE: 5000

--- Preparando setup para: 4 RÓTULOS POR CLASSE ---
Setup completo para 4 rótulos.
  Imagens de Treino (Rotuladas): 40
  Imagens de Treino (Não-Rotuladas): 50000
  Imagens de VALIDAÇÃO: 5000
  Imagens de TESTE: 5000

--- Preparando setup para: 25 RÓTULOS POR CLASSE ---
Setup completo para 25 rótulos.
  Imagens de Treino (Rotuladas): 250
  Imagens de Treino (Não-Rotuladas): 50000
  Imagens de VALIDAÇÃO: 5000
  Imagens de TESTE: 5000

--- Preparando setup para: 400 RÓTULOS POR CLASSE ---
Setup completo para 400 rótulos.
  Imagens de Treino (Rotuladas): 4000
  Imagens de Treino (Não-Rotuladas): 50000
  Imagens de VALIDAÇÃO: 5000
  Imagens de TESTE: 5000



# Para **testes** *só de 1*

In [ ]:
#LABEL_COUNTS = [400]
#NUM_EPOCHS = 5
#LEARNING_RATE = 0.01

# EXPERIMENTOS FIXMATCH

---

Rodando o modelo, treinando a partir de um modelo já existente ou usando um .pth anterior (fizemos isso bastante para rodar em várias contas do COLAB)

In [ ]:
fixmatch_results = {}

print("EXPERIMENTOS FIXMATCH")
print("=" * 40)

for num_labels in LABEL_COUNTS:
    print(f"\n--- Experimento FixMatch: {num_labels} Rótulos por Classe ---")

    # Pega os loaders
    loaders = data_setups[num_labels]

    # Pega um modelo pronto pth se houver
    model_filename = f"fixmatch_model_{num_labels}_labels.pth"
    model_fixmatch = create_model().to(device)

    if os.path.exists(model_filename):
        print(f"Modelo salvo encontrado ({model_filename}). Carregando pesos...")
        try:
            model_fixmatch.load_state_dict(torch.load(model_filename))
            print("Pesos carregados com sucesso. Continuando o treinamento.")
        except Exception as e:
            print(f"Erro ao carregar o modelo: {e}. Treinando do zero.")
    else:
        print(f"Modelo não encontrado. Treinando do zero...")

    optimizer = optim.SGD(model_fixmatch.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4, nesterov=True)

    # Treinando o modelo (função criada lá no começo)
    history = train_model(
        model=model_fixmatch,
        labeled_loader=loaders['labeled_loader'],
        criterion_supervised=loss_fn_supervised,
        unlabeled_loader=loaders['unlabeled_loader'],
        criterion_unsupervised=loss_fn_unsupervised,
        optimizer=optimizer,
        num_epochs=NUM_EPOCHS,
        TAU=TAU,
        LAMBDA_U=LAMBDA_U,
        val_loader=loaders['val_loader'],
        device=device
    )

    # Plota o histórico de treino/validação
    plot_history(history, f"FixMatch - {num_labels} Rótulos por Classe")

    # Avaliamos aqui no conjunto de TESTE (uma única vez, no final)
    final_test_loss, final_test_acc = evaluate_model(
        model_fixmatch,
        loaders['test_loader'],
        loss_fn_supervised,
        device
    )
    print(f"✅ Acurácia Final no Teste (FixMatch, {num_labels} rótulos): {final_test_acc:.2f}%")

    # Salvando os resultados
    fixmatch_results[num_labels] = {
        'history': history,
        'accuracy': final_test_acc
    }

    # Baixando o modelo treinado
    #torch.save(model_fixmatch.state_dict(), model_filename)
    #print(f"Modelo salvo em: {model_filename}")
    #files.download(model_filename)
    #print(f"Iniciando download de {model_filename}...")

# BASELINE
---
Experimentos só com as imagens com label (mesmo número de imagens de label que a anterior, porém sem as sem label). Esperamos um resultado pior


In [ ]:
supervised_results = {}

print("\n\nEXPERIMENTOS BASELINE (SUPERVISIONADO)")
print("=" * 40)

for num_labels in LABEL_COUNTS:
    print(f"\n--- Experimento Baseline: {num_labels} Rótulos por Classe ---")

    # Pega os loaders
    loaders = data_setups[num_labels]

    # Pega um modelo pronto pth se houver
    model_filename = f"baseline_model_{num_labels}_labels.pth"

    model_baseline = create_model().to(device)

    if os.path.exists(model_filename):
        print(f"Modelo salvo encontrado ({model_filename}). Carregando pesos...")
        try:
            model_baseline.load_state_dict(torch.load(model_filename))
            print("Pesos carregados com sucesso. Continuando o treinamento.")
        except Exception as e:
            print(f"Erro ao carregar o modelo: {e}. Treinando do zero.")
    else:
        print(f"Modelo não encontrado. Treinando do zero...")

    optimizer = optim.SGD(model_baseline.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4, nesterov=True)

    # Treina o modelo
    history = train_supervised_only(
        model=model_baseline,
        unlabeled_loader=loaders['unlabeled_loader'],
        labeled_loader=loaders['labeled_loader'],
        criterion_supervised=loss_fn_supervised,
        optimizer=optimizer,
        num_epochs=NUM_EPOCHS,
        val_loader=loaders['val_loader'],
        device=device
    )

    # Plota o histórico de treino/validação
    plot_history(history, f"Baseline (Supervisionado) - {num_labels} Rótulos por Classe")

    # Avaliando no conjunto de TESTE (uma única vez, no final)
    final_test_loss, final_test_acc = evaluate_model(
        model_baseline,
        loaders['test_loader'],
        loss_fn_supervised,
        device
    )
    print(f"Acurácia Final no Teste (Baseline, {num_labels} rótulos): {final_test_acc:.2f}%")

    # Salvando os resultados
    supervised_results[num_labels] = {
        'history': history,
        'accuracy': final_test_acc
    }

    # Salvando e baixando o modelo treinado
    torch.save(model_baseline.state_dict(), model_filename)
    print(f"Modelo salvo em: {model_filename}")
    files.download(model_filename)
    print(f"Iniciando download de {model_filename}...")

# Avaliação no Teste

---

Pega os loaders de teste do DataModule do CIFAR10 e faz a avaliação do modelo FixMatch nosso e do Baseline

In [ ]:
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COUNTS = [1, 4, 25, 400]

print("\nPreparando o loader de teste...")
try:
    eval_data_module = CIFAR10DataModule()
    eval_data_module.setup()
    test_loader = eval_data_module.get_test_loader()
    print("Loader de teste pronto.")
except NameError:
    print("\nA classe 'CIFAR10DataModule' não está definida.")
    raise

# Recriando os Dicionários de Resultados
fixmatch_results = {}
supervised_results = {}

print("\nIniciando avaliação dos modelos carregados (via upload manual)...")
print("=" * 40)

for num_labels in LABEL_COUNTS:
    print(f"\n--- Avaliando Experimento: {num_labels} Rótulos por Classe ---")

    # =========================
    # Avaliando Modelo FixMatch
    # =========================
    model_fixmatch = create_model().to(device)
    filename_fixmatch = f"fixmatch_model_{num_labels}_labels.pth"

    # Pega o model .pth pronto (se tiver)
    if os.path.exists(filename_fixmatch):
        model_fixmatch.load_state_dict(torch.load(filename_fixmatch))
        acc_fixmatch = evaluate_model(model_fixmatch, test_loader, loss_fn_supervised, device)
        fixmatch_results[num_labels] = {'accuracy': acc_fixmatch}
    else:
        print(f"  ERRO: Arquivo '{filename_fixmatch}' não encontrado.")
        fixmatch_results[num_labels] = {'accuracy': 'N/A'}

    # =========================
    # Avaliando Modelo Baseline
    # =========================
    model_baseline = create_model().to(device)
    filename_baseline = f"baseline_model_{num_labels}_labels.pth"

    # Pega o model .pth pronto (se tiver)
    if os.path.exists(filename_baseline):
        model_baseline.load_state_dict(torch.load(filename_baseline))
        acc_baseline = evaluate_model(model_baseline, test_loader, loss_fn_supervised, device)
        supervised_results[num_labels] = {'accuracy': acc_baseline}
    else:
        print(f"  ERRO: Arquivo '{filename_baseline}' não encontrado.")
        supervised_results[num_labels] = {'accuracy': 'N/A'}

print(fixmatch_results)
print(supervised_results)

# Plots de Comparação

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

results_data = []

for num_labels in LABEL_COUNTS:

    sup_acc_tuple = supervised_results.get(num_labels, {}).get('accuracy', (None, 'N/A'))
    fix_acc_tuple = fixmatch_results.get(num_labels, {}).get('accuracy', (None, 'N/A'))

    results_data.append({
        'Rótulos por Classe': num_labels,
        'Total de Rótulos': num_labels * 10,
        'Acurácia (Baseline)': sup_acc_tuple[1],
        'Acurácia (FixMatch)': fix_acc_tuple[1]
    })

df_results = pd.DataFrame(results_data)
df_results['Acurácia (Baseline)'] = pd.to_numeric(df_results['Acurácia (Baseline)'], errors='coerce')
df_results['Acurácia (FixMatch)'] = pd.to_numeric(df_results['Acurácia (FixMatch)'], errors='coerce')

print("\n\n📊 === RESULTADOS FINAIS DOS EXPERIMENTOS === 📊")
print(df_results.to_markdown(index=False))

plt.figure(figsize=(10, 6))
plt.plot(df_results['Total de Rótulos'], df_results['Acurácia (Baseline)'], marker='o', linestyle='--', label='Baseline (Só Supervisionado)')
plt.plot(df_results['Total de Rótulos'], df_results['Acurácia (FixMatch)'], marker='o', linestyle='-', label='FixMatch (Semi-Supervisionado)')
plt.xscale('log') # Melhor visão para o eixo x
plt.xlabel('Total de Rótulos (Escala Log)')
plt.ylabel('Acurácia no Teste (%)')
plt.title('Desempenho do FixMatch vs. Baseline Supervisionado')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

# Matriz de confusão e as Imagens de exemplo

In [ ]:
# Carrega o modelo e gera predições
def get_predictions(model_path, test_loader):
    model = create_model().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    all_preds = []
    all_labels = []
    all_images = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(F.softmax(outputs, dim=1), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_images.extend(images.cpu())

    return np.array(all_preds), np.array(all_labels), all_images

# Plota matriz de confusão
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cifar_classes, yticklabels=cifar_classes)
    plt.xlabel("Predito")
    plt.ylabel("Verdadeiro")
    plt.title(title)
    plt.show()

# Visualiza predições corretas e incorretas
def show_predictions(images, y_true, y_pred, class_names, n=16):
    plt.figure(figsize=(12, 8))
    indices = np.random.choice(len(images), n, replace=False)
    for i, idx in enumerate(indices):
        img = images[idx].permute(1, 2, 0) * torch.tensor([0.2023, 0.1994, 0.2010]) + torch.tensor([0.4914, 0.4822, 0.4465])
        img = torch.clamp(img, 0, 1)
        true_label = class_names[y_true[idx]]
        pred_label = class_names[y_pred[idx]]
        plt.subplot(4, 4, i + 1)
        plt.imshow(img)
        color = "green" if true_label == pred_label else "red"
        plt.title(f"T:{true_label}\nP:{pred_label}", color=color, fontsize=8)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Classes do CIFAR-10
cifar_classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Geração das figuras
for num_labels in [1, 4, 25, 400]:
    model_path = f"fixmatch_model_{num_labels}_labels.pth"
    loaders = data_setups[num_labels]
    test_loader = loaders['test_loader']

    print(f"\n🔹 Avaliando modelo com {num_labels} rótulos por classe...")
    preds, labels, imgs = get_predictions(model_path, test_loader)

    # 1️⃣ Matriz de confusão
    plot_confusion_matrix(labels, preds, f"FixMatch ({num_labels} rótulos/classe)")

    # 2️⃣ Amostras de predições
    show_predictions(imgs, labels, preds, cifar_classes, n=16)


In [ ]:
# Salvar o modelo
#PATH = "model10epoch.pth"
#torch.save(model_init, PATH)

# model_carregado = torch.load(PATH)
# model_carregado.eval()